# XQuery 4.0 CSV Functions

The [QT4 specification](https://qt4cg.org/specifications/xpath-functions-40/Overview.html#csv-functions) defines a family of CSV functions for parsing and generating CSV data. These complement the `method="csv"` serialization with round-trip capabilities.

> **Status**: These functions are planned for eXist-db's XQuery 4.0 implementation. The examples below show the expected behavior per the QT4 spec.

## fn:csv-to-arrays

Parses CSV text into a sequence of string arrays — one array per row:

In [ ]:
(: When implemented, this will work: :)
(:
fn:csv-to-arrays("Name,Age,City
Alice,30,New York
Bob,25,Portland")
:)

(: Expected result:
   (["Name","Age","City"], ["Alice","30","New York"], ["Bob","25","Portland"])
:)

(: For now, simulate with: :)
let $csv := "Name,Age,City&#10;Alice,30,New York&#10;Bob,25,Portland"
let $lines := tokenize($csv, "&#10;")
return
    array {
        for $line in $lines
        return array { tokenize($line, ",") }
    }

## fn:parse-csv

Parses CSV into a structured record with column names and typed data:

In [ ]:
(: When implemented:
fn:parse-csv("Name,Age,City
Alice,30,New York
Bob,25,Portland",
map { "header": true() })
:)

(: Returns a record:
   record {
     columns: ("Name", "Age", "City"),
     rows: (["Alice", "30", "New York"], ["Bob", "25", "Portland"])
   }
:)

(: Simulate the concept: :)
let $csv := "Name,Age,City&#10;Alice,30,New York&#10;Bob,25,Portland"
let $lines := tokenize($csv, "&#10;")
let $header := tokenize(head($lines), ",")
let $rows :=
    for $line in tail($lines)
    return array { tokenize($line, ",") }
return map {
    "columns": array { $header },
    "column-count": count($header),
    "row-count": count($rows),
    "rows": array { $rows }
}

## fn:csv

Serializes data as CSV text — the inverse of [`fn:parse-csv`]({docs}/functions/fn/parse-csv):

In [ ]:
(: When implemented:
fn:csv(
    (["Alice", "30"], ["Bob", "25"]),
    map { "header": ("Name", "Age") }
)
:)

(: For now, use method="csv" serialization: :)
serialize([
    ["Name", "Age"],
    ["Alice", "30"],
    ["Bob", "25"]
], map { "method": "csv", "csv.quotes": false() })

## CSV Options (QT4 Spec)

The QT4 spec defines these options for CSV functions:

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `field-delimiter` | string | `,` | Character separating fields |
| `row-delimiter` | string | `\n` | Line ending |
| `quote-character` | string | `"` | Quoting character |
| `header` | boolean | `false` | First row is header |
| `trim-whitespace` | boolean | `false` | Trim leading/trailing whitespace from fields |
| `number-of-columns` | integer | (auto) | Expected column count |
| `select-columns` | integer* | all | Which columns to include |

These map closely to the `csv.*` serialization parameters already supported by `method="csv"`.

## Round-Trip Pattern

The full round-trip from CSV text → XQuery data → CSV text:

In [ ]:
(: Step 1: Parse CSV into arrays (simulated) :)
let $input := "Name,Score&#10;Alice,95&#10;Bob,87"
let $lines := tokenize($input, "&#10;")
let $arrays :=
    for $line in $lines
    return array { tokenize($line, ",") }

(: Step 2: Process — filter high scores :)
let $header := head($arrays)
let $filtered :=
    for $row in tail($arrays)
    where xs:integer($row(2)) >= 90
    return $row

(: Step 3: Serialize back to CSV :)
return serialize(
    array { $header, $filtered },
    map { "method": "csv", "csv.quotes": false() }
)

## Working with Real CSV Files

### Load CSV from database

In [ ]:
(: Store a CSV file in the database, then parse it: :)
let $csv-text := util:binary-to-string(
    util:binary-doc("/db/data/employees.csv")
)
let $lines := tokenize($csv-text, "&#10;")
let $header := tokenize(head($lines), ",")
return
    <table>
        <thead><tr>{
            for $col in $header return <th>{$col}</th>
        }</tr></thead>
        <tbody>{
            for $line in tail($lines)
            let $fields := tokenize($line, ",")
            return
                <tr>{
                    for $field in $fields return <td>{$field}</td>
                }</tr>
        }</tbody>
    </table>

### Convert XML collection to CSV

In [ ]:
(: Export all books as CSV :)
(:
let $books := collection("/db/library")//book
return serialize(
    for $b in $books
    return map {
        "title": $b/title/string(),
        "author": $b/author/string(),
        "year": $b/year/string(),
        "isbn": $b/isbn/string()
    },
    map { "method": "csv", "csv.header": true(), "csv.quotes": false() }
)
:)
"(Run against a collection with book documents)"

## Comparison: eXist vs BaseX vs Saxon

| Feature | eXist-db | BaseX | Saxon |
|---------|----------|-------|-------|
| `method="csv"` | Yes | Yes | No |
| [`fn:parse-csv`]({docs}/functions/fn/parse-csv) | Planned | Yes (13+) | Yes (12+) |
| [`fn:csv-to-arrays`]({docs}/functions/fn/csv-to-arrays) | Planned | Yes (13+) | Yes (12+) |
| [`fn:csv`]({docs}/functions/fn/csv) | Planned | Yes (13+) | Yes (12+) |
| Custom delimiters | Yes | Yes | N/A |
| Map input | Yes | Yes | N/A |
| XML table input | Yes | Yes | N/A |
| Header support | Yes | Yes | N/A |